# MGMT298D: Science and Strategy of AI
### Week 4A - Multi-Armed Bandits
### Application: E-commerce Price Optimization

## Import Libraries and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## The Pricing Environment

We have 5 possible prices for a product. Higher prices mean fewer buyers but more revenue per sale.

**The challenge:** We don't know the true conversion rates! We must learn them through experimentation.

In [ ]:
# The "true" environment (pretend you don't know this!)
PRICES = [5, 10, 15, 20, 25]
TRUE_CONVERSION_RATES = [0.50, 0.35, 0.22, 0.12, 0.05]

def simulate_sale(price_index):
    """Simulate one customer. Returns revenue (price if they buy, 0 if not)."""
    bought = np.random.random() < TRUE_CONVERSION_RATES[price_index]
    return PRICES[price_index] if bought else 0

# What's the expected revenue at each price?
expected_revenues = [p * r for p, r in zip(PRICES, TRUE_CONVERSION_RATES)]
print("Expected revenue per customer at each price:")
for p, r, e in zip(PRICES, TRUE_CONVERSION_RATES, expected_revenues):
    print(f"  ${p}: {r:.0%} conversion → ${e:.2f} expected")
print(f"\nOptimal price: ${PRICES[expected_revenues.index(max(expected_revenues))]}")

## Strategy A: Random Selection

In [ ]:
def run_random(n_customers):
    """Pick prices randomly."""
    cumulative = 0
    history = []
    counts = [0] * 5
    
    for _ in range(n_customers):
        choice = np.random.randint(5)
        revenue = simulate_sale(choice)
        cumulative += revenue
        history.append(cumulative)
        counts[choice] += 1
    
    return history, counts

random_history, random_counts = run_random(1000)
print(f"Random strategy: ${random_history[-1]:,} total revenue")

## Strategy B: Epsilon-Greedy

Usually pick the best-so-far, but explore randomly ε% of the time.

In [ ]:
def run_epsilon_greedy(n_customers, epsilon=0.1):
    """Epsilon-greedy: explore with probability epsilon, exploit otherwise."""
    counts = [0] * 5
    total_rewards = [0] * 5
    cumulative = 0
    history = []
    
    for _ in range(n_customers):
        # Try each price once first
        if min(counts) == 0:
            choice = counts.index(0)
        elif np.random.random() < epsilon:
            choice = np.random.randint(5)  # Explore
        else:
            avgs = [total_rewards[i] / counts[i] for i in range(5)]
            choice = avgs.index(max(avgs))  # Exploit
        
        revenue = simulate_sale(choice)
        counts[choice] += 1
        total_rewards[choice] += revenue
        cumulative += revenue
        history.append(cumulative)
    
    return history, counts

eg_history, eg_counts = run_epsilon_greedy(1000, epsilon=0.1)
print(f"ε-Greedy strategy: ${eg_history[-1]:,} total revenue")

## Strategy C: Upper Confidence Bound (UCB)

Pick the price with highest (average + uncertainty bonus). Automatically explores uncertain options!

In [ ]:
def run_ucb(n_customers, confidence=2.0):
    """UCB: balance exploitation with exploration via confidence bounds."""
    counts = [0] * 5
    total_rewards = [0] * 5
    cumulative = 0
    history = []
    
    for t in range(n_customers):
        # Try each price once first
        if min(counts) == 0:
            choice = counts.index(0)
        else:
            ucb_scores = []
            for i in range(5):
                avg = total_rewards[i] / counts[i]
                bonus = confidence * np.sqrt(np.log(t + 1) / counts[i])
                ucb_scores.append(avg + bonus)
            choice = ucb_scores.index(max(ucb_scores))
        
        revenue = simulate_sale(choice)
        counts[choice] += 1
        total_rewards[choice] += revenue
        cumulative += revenue
        history.append(cumulative)
    
    return history, counts

ucb_history, ucb_counts = run_ucb(1000, confidence=2.0)
print(f"UCB strategy: ${ucb_history[-1]:,} total revenue")

## Strategy Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Revenue over time
axes[0].plot(random_history, label="Random", alpha=0.7)
axes[0].plot(eg_history, label="ε-Greedy (ε=0.1)", linewidth=2)
axes[0].plot(ucb_history, label="UCB (c=2.0)", linewidth=2)
axes[0].set_xlabel("Customer #")
axes[0].set_ylabel("Cumulative Revenue ($)")
axes[0].set_title("Which Strategy Earns the Most?")
axes[0].legend()

# Right: How often each price was chosen
x = np.arange(5)
width = 0.25
axes[1].bar(x - width, random_counts, width, label="Random", alpha=0.7)
axes[1].bar(x, eg_counts, width, label="ε-Greedy", alpha=0.7)
axes[1].bar(x + width, ucb_counts, width, label="UCB", alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(["$5", "$10", "$15", "$20", "$25"])
axes[1].set_xlabel("Price")
axes[1].set_ylabel("Times Chosen")
axes[1].set_title("Which Prices Did Each Strategy Try?")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final Revenue after 1,000 customers:")
print(f"  Random:    ${random_history[-1]:,}")
print(f"  ε-Greedy:  ${eg_history[-1]:,}")
print(f"  UCB:       ${ucb_history[-1]:,}")

## Effect of Epsilon on Exploration

In [ ]:
epsilons = [0.0, 0.05, 0.1, 0.2, 0.5]

fig, ax = plt.subplots(figsize=(10, 5))

for eps in epsilons:
    history, _ = run_epsilon_greedy(1000, epsilon=eps)
    ax.plot(history, label=f"ε = {eps}", linewidth=2)

ax.set_xlabel("Customer #")
ax.set_ylabel("Cumulative Revenue ($)")
ax.set_title("Effect of Epsilon on Learning")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Effect of UCB Confidence Parameter

In [ ]:
confidences = [0.5, 1.0, 2.0, 5.0]

fig, ax = plt.subplots(figsize=(10, 5))

for c in confidences:
    history, _ = run_ucb(1000, confidence=c)
    ax.plot(history, label=f"c = {c}", linewidth=2)

ax.set_xlabel("Customer #")
ax.set_ylabel("Cumulative Revenue ($)")
ax.set_title("Effect of UCB Confidence Parameter")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()